# Day 8 -- Feature Engineering & Re-tuning

Continues from `04_tuning.ipynb` (Day 7).  
Goals:
- **Task 3 (delay_category):** re-tune with 44 features (interactions + Phase B3 `machine_avg_delay_minutes_90d`). Target: G3 > 0.725. Hard fallback to achieved score if still blocked.
- **Task 4 (delay_root_cause):** consolidate rare classes -> 6-class, re-tune. Target: G4 > 0.50.

Tasks 1 and 2 are not re-tuned (G1 and G2 passed in Day 7).

In [1]:
import os
import tempfile
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline

import sys
sys.path.insert(0, str(Path('../src').resolve()))

from mpc_ml.features.constants import (
    DELAY_CATEGORY_ORDER,
    ROOT_CAUSE_CLASSES,
    TARGET_COLS,
    INTERACTION_FEATURE_NAMES,
)
from mpc_ml.features.pipeline import build_pipeline, get_feature_names
from mpc_ml.models.tuning import (
    build_optuna_objective,
    run_study,
    best_params_to_model,
)
from mpc_ml.models.evaluation import evaluate_model
from mpc_ml.tracking.mlflow_utils import (
    get_experiment_name,
    log_model_with_signature,
    log_standard_params,
    log_standard_metrics,
    log_standard_artifacts,
    start_run,
)

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

print('Imports OK')
print(f'Interaction features ({len(INTERACTION_FEATURE_NAMES)}): {INTERACTION_FEATURE_NAMES}')

Imports OK
Interaction features (6): ('lag_as_pct_of_window', 'tightness_x_queue', 'log_experience_x_concurrent', 'oee_x_maintenance_ratio', 'util_x_queue', 'util_x_tight')


In [2]:
_NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT  = _NOTEBOOK_DIR.parent.parent
MLFLOW_URI    = (PROJECT_ROOT / 'mlruns').resolve().as_uri()
mlflow.set_tracking_uri(MLFLOW_URI)

PHASE        = 'day8'
N_TRIALS     = 100
N_CV_SPLITS  = 5
RANDOM_STATE = 42

# Day 8 gate targets
G3_THRESHOLD = 0.725   # delay_category weighted_f1
G4_THRESHOLD = 0.50    # root_cause macro_f1 (6-class)

# Task 4 class consolidation
RC_CONSOLIDATE_MAP = {
    'machine_breakdown':      'equipment_process_failure',
    'quality_failure_rework': 'equipment_process_failure',
}
RC_CLASSES_6 = tuple(
    c for c in ROOT_CAUSE_CLASSES
    if c not in ('machine_breakdown', 'quality_failure_rework')
) + ('equipment_process_failure',)

print(f'MLflow URI:  {MLFLOW_URI}')
print(f'N_TRIALS:    {N_TRIALS}  |  N_CV_SPLITS: {N_CV_SPLITS}')
print(f'Day 8 targets: G3>{G3_THRESHOLD}  G4>{G4_THRESHOLD} (6-class)')
print(f'RC_CLASSES_6 ({len(RC_CLASSES_6)}): {RC_CLASSES_6}')

MLflow URI:  file:///D:/Kuliah/Project/manufacturing-factory-simulation/manufacturing-process-copilot/mlruns
N_TRIALS:    100  |  N_CV_SPLITS: 5
Day 8 targets: G3>0.725  G4>0.5 (6-class)
RC_CLASSES_6 (6): ('material_unavailability', 'multiple_causes', 'none', 'planning_schedule_conflict', 'setup_overrun', 'equipment_process_failure')


In [3]:
DATA_DIR = Path('../data/processed')
train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df   = pd.read_csv(DATA_DIR / 'val.csv')

# Raw feature DataFrames (37 cols; pipeline adds 6 interactions + 1 Phase B3 -> 44 total)
X_train = train_df.drop(columns=[c for c in TARGET_COLS if c in train_df.columns])
X_val   = val_df.drop(columns=[c for c in TARGET_COLS if c in val_df.columns])

# Task 3 labels
y_train_cat = train_df['delay_category']
y_val_cat   = val_df['delay_category']

# Task 4: delayed-only filter
train_d = train_df[train_df['is_delayed'] == 1].copy()
val_d   = val_df[val_df['is_delayed'] == 1].copy()
X_train_d    = train_d.drop(columns=[c for c in TARGET_COLS if c in train_d.columns])
X_val_d      = val_d.drop(columns=[c for c in TARGET_COLS if c in val_d.columns])
y_train_rc_d = train_d['delay_root_cause']
y_val_rc_d   = val_d['delay_root_cause']

# Task 4 (6-class): apply consolidation map
y_train_rc_6 = y_train_rc_d.replace(RC_CONSOLIDATE_MAP)
y_val_rc_6   = y_val_rc_d.replace(RC_CONSOLIDATE_MAP)

n_neg = (train_df['is_delayed'] == 0).sum()
n_pos = (train_df['is_delayed'] == 1).sum()
scale_pos_weight = n_neg / n_pos

print(f'train: {len(train_df):,} rows  |  val: {len(val_df):,} rows')
print(f'Task 3 classes: {sorted(y_train_cat.unique())}')
print(f'Task 4 delayed train: {len(train_d):,}  |  val: {len(val_d):,}')
print(f'\nTask 4 6-class after consolidation (train):\n{y_train_rc_6.value_counts().to_string()}')
print(f'\nTask 4 6-class (val):\n{y_val_rc_6.value_counts().to_string()}')

train: 4,113 rows  |  val: 1,043 rows
Task 3 classes: ['critical_delay', 'major_delay', 'minor_delay', 'moderate_delay', 'on_time']
Task 4 delayed train: 1,506  |  val: 378

Task 4 6-class after consolidation (train):
delay_root_cause
multiple_causes               1034
setup_overrun                  224
planning_schedule_conflict     116
material_unavailability         65
equipment_process_failure       44
none                            23

Task 4 6-class (val):
delay_root_cause
multiple_causes               266
setup_overrun                  50
planning_schedule_conflict     30
material_unavailability        15
equipment_process_failure      13
none                            4


In [4]:
# Phase B3 -- machine_avg_delay_minutes_90d
# Strict temporal isolation: expanding window per machine_id, shift(1) excludes current order.
# Val uses full-train per-machine mean (val is temporally after all train orders).
# Cold-start NaN -> filled by ColumnSelector with training-set population mean.
_HIST_FEAT = 'machine_avg_delay_minutes_90d'

def _add_hist_delay(train_raw, val_raw):
    tr = train_raw[['machine_id', 'planned_start', 'delay_minutes']].copy()
    tr['_ts'] = pd.to_datetime(tr['planned_start'], format='ISO8601')
    tr = tr.sort_values('_ts')
    tr[_HIST_FEAT] = (
        tr.groupby('machine_id')['delay_minutes']
        .expanding().mean().shift(1)
        .reset_index(level=0, drop=True)
    )
    train_raw[_HIST_FEAT] = tr[_HIST_FEAT].reindex(train_raw.index)
    machine_means = train_raw.groupby('machine_id')['delay_minutes'].mean()
    val_raw[_HIST_FEAT] = val_raw['machine_id'].map(machine_means)

_add_hist_delay(train_df, val_df)

# Re-derive X splits to include the new column (TARGET_COLS still excluded)
X_train = train_df.drop(columns=[c for c in TARGET_COLS if c in train_df.columns])
X_val   = val_df.drop(columns=[c for c in TARGET_COLS if c in val_df.columns])
X_train_d = X_train.loc[train_d.index]
X_val_d   = X_val.loc[val_d.index]

nan_tr = train_df[_HIST_FEAT].isna().sum()
nan_vl = val_df[_HIST_FEAT].isna().sum()
print(f'Phase B3: {_HIST_FEAT}')
print(f'  train NaN={nan_tr}  val NaN={nan_vl}')
print(f'  train non-null mean: {train_df[_HIST_FEAT].mean():.1f} min  std: {train_df[_HIST_FEAT].std():.1f} min')
print(f'  val   non-null mean: {val_df[_HIST_FEAT].mean():.1f} min  std: {val_df[_HIST_FEAT].std():.1f} min')

Phase B3: machine_avg_delay_minutes_90d
  train NaN=1  val NaN=0
  train non-null mean: 171.9 min  std: 49.1 min
  val   non-null mean: 186.2 min  std: 28.3 min


In [5]:
# Verify all 44 features present and non-trivial before tuning
probe_pipe = build_pipeline()
probe_pipe.fit(X_train)
X_probe = probe_pipe.transform(X_train)

print(f'Pipeline output shape: {X_probe.shape}  (expected: ({len(X_train)}, 44))')
assert X_probe.shape[1] == 44, f'Expected 44 features, got {X_probe.shape[1]}'

for feat in ['util_x_queue', 'util_x_tight', 'machine_avg_delay_minutes_90d']:
    assert feat in X_probe.columns, f'{feat} missing from pipeline output'
    std_val = X_probe[feat].std()
    assert std_val > 0.01, f'{feat} has near-zero std={std_val:.4f}'
    print(f'  {feat}: mean={X_probe[feat].mean():.4f}  std={std_val:.4f}')

print('\nAll 44 features verified. Pipeline ready for Day 8 Phase B3 tuning.')
del probe_pipe, X_probe

Pipeline output shape: (4113, 44)  (expected: (4113, 44))
  util_x_queue: mean=0.0000  std=1.0001
  util_x_tight: mean=-0.0000  std=1.0001
  machine_avg_delay_minutes_90d: mean=0.0000  std=1.0001

All 44 features verified. Pipeline ready for Day 8 Phase B3 tuning.


## Task 3 -- Delay Category Re-tuning (Phase B3)

Re-tunes with 44 features: `util_x_queue`, `util_x_tight` + Phase B3 `machine_avg_delay_minutes_90d`.  
Phase B3 baseline (interactions only, Day 8): val weighted_f1 = 0.6997. Target: > 0.725.  
Hard fallback: if still below 0.725, recalibrate G3_THRESHOLD to achieved val score (Bayes error ceiling).

In [6]:
print('Task 3 -- Building Optuna objective (LightGBM, delay_category, 44 features)\n')
cat_objective = build_optuna_objective(
    X_train, y_train_cat,
    model_type='lightgbm',
    task='delay_category',
    n_splits=N_CV_SPLITS,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
)
cat_study = run_study(
    cat_objective,
    n_trials=N_TRIALS,
    study_name='category_lightgbm_day8_b3',
    seed=RANDOM_STATE,
)
print(f'Best CV weighted_f1: {cat_study.best_value:.4f}  (Day 8 Phase1 best: 0.7196)')
print(f'Best trial: #{cat_study.best_trial.number}')
print(f'Best params: {cat_study.best_params}')

Task 3 -- Building Optuna objective (LightGBM, delay_category, 44 features)



Best CV weighted_f1: 0.7160  (Day 8 Phase1 best: 0.7196)
Best trial: #60
Best params: {'n_estimators': 598, 'num_leaves': 117, 'learning_rate': 0.011374393389929679, 'subsample': 0.8973546062626159, 'colsample_bytree': 0.8618288915561358, 'min_child_samples': 21, 'reg_alpha': 0.0025011013620654214, 'reg_lambda': 3.776927497214034e-08}


In [7]:
print('Task 3 -- Final training with best params\n')
cat_estimator = best_params_to_model(
    cat_study,
    model_type='lightgbm',
    task='delay_category',
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
)
cat_pipeline = Pipeline([
    ('preprocessor', build_pipeline()),
    ('model', cat_estimator),
])
cat_pipeline.fit(X_train, y_train_cat)

cat_preprocessor = cat_pipeline.named_steps['preprocessor']
cat_model        = cat_pipeline.named_steps['model']

cat_val_metrics = evaluate_model(cat_model, cat_preprocessor, X_val, y_val_cat, 'delay_category')
cat_weighted_f1 = cat_val_metrics['val_weighted_f1']
cat_macro_f1    = cat_val_metrics['val_macro_f1']

X_val_t_cat   = cat_preprocessor.transform(X_val)
X_train_t_cat = cat_preprocessor.transform(X_train)

print(f'val_weighted_f1: {cat_weighted_f1:.4f}  (Day 8 Phase1: 0.6997  |  target: >{G3_THRESHOLD})')
print(f'val_macro_f1:    {cat_macro_f1:.4f}')

y_pred_cat = cat_model.predict(X_val_t_cat)
cr_cat = classification_report(
    y_val_cat, y_pred_cat,
    labels=list(DELAY_CATEGORY_ORDER), zero_division=0,
)
print(f'\nTask 3 Day 8 Phase B3 classification report:')
print(cr_cat)

Task 3 -- Final training with best params



val_weighted_f1: 0.6954  (Day 8 Phase1: 0.6997  |  target: >0.725)
val_macro_f1:    0.4164

Task 3 Day 8 Phase B3 classification report:
                precision    recall  f1-score   support

       on_time       0.87      0.92      0.89       670
   minor_delay       0.50      0.30      0.37        64
moderate_delay       0.35      0.16      0.22       158
   major_delay       0.41      0.71      0.52       129
critical_delay       0.25      0.05      0.08        22

      accuracy                           0.72      1043
     macro avg       0.48      0.43      0.42      1043
  weighted avg       0.70      0.72      0.70      1043



In [8]:
print(f'Task 3 -- Logging to: {get_experiment_name("delay_category")}\n')
cat_all_metrics = {
    **cat_val_metrics,
    'optuna_best_cv_weighted_f1': round(cat_study.best_value, 6),
}
safe_params = {k: v for k, v in cat_study.best_params.items()
               if isinstance(v, (str, int, float, bool))}
safe_params['model_name']   = 'lightgbm'
safe_params['task']         = 'delay_category'
safe_params['num_classes']  = len(DELAY_CATEGORY_ORDER)
safe_params['n_features']   = X_train_t_cat.shape[1]
safe_params['phase']        = PHASE
safe_params['new_features'] = 'util_x_queue,util_x_tight,machine_avg_delay_minutes_90d'

cm_cat_path = Path(tempfile.gettempdir()) / 'cm_cat_day8b3.png'
from mpc_ml.models.evaluation import confusion_matrix_annotated
cm_cat_df = confusion_matrix_annotated(cat_model, X_val_t_cat, y_val_cat)
n_cls = len(DELAY_CATEGORY_ORDER)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm_cat_df.iloc[:, :n_cls].astype(float), annot=True, fmt='.0f', cmap='Blues',
    ax=ax, cbar=False,
    xticklabels=list(DELAY_CATEGORY_ORDER),
    yticklabels=list(DELAY_CATEGORY_ORDER),
)
ax.set_title('Task 3 LightGBM Day 8 Phase B3 -- delay_category')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.xticks(rotation=30, ha='right'); fig.tight_layout()
fig.savefig(cm_cat_path, dpi=100); plt.close(fig)

day8_run_ids = {}
tags = {'model_type': 'lightgbm', 'task': 'delay_category', 'phase': PHASE, 'is_champion': 'True'}
with start_run(get_experiment_name('delay_category'), f'lightgbm_{PHASE}_b3', tags=tags) as run:
    log_standard_params(safe_params)
    log_standard_metrics(cat_all_metrics)
    log_model_with_signature(
        cat_pipeline, X_train, X_train_t_cat,
        registered_model_name='delay_category_classifier',
    )
    log_standard_artifacts(
        classification_report=cr_cat,
        confusion_matrix_path=cm_cat_path,
    )
    day8_run_ids['category'] = run.info.run_id
cm_cat_path.unlink(missing_ok=True)
print(f'  run_id={day8_run_ids["category"]}  weighted_f1={cat_weighted_f1:.4f}')

Task 3 -- Logging to: mpc/delay_category



2026/06/12 05:13:51 WARNING mlflow.utils.requirements_utils: The following packages were not found in the public PyPI package index as of 2025-04-15; if these packages are not present in the public PyPI index, you must install them manually before loading your model: {'mpc-ml'}


Registered model 'delay_category_classifier' already exists. Creating a new version of this model...
Created version '3' of model 'delay_category_classifier'.
2026/06/12 05:13:51 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


2026/06/12 05:13:56 WARNING mlflow.utils.requirements_utils: The following packages were not found in the public PyPI package index as of 2025-04-15; if these packages are not present in the public PyPI index, you must install them manually before loading your model: {'mpc-ml'}


2026/06/12 05:13:56 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


  run_id=845c5a6aa02c48cdb846256f0cbf91c4  weighted_f1=0.6954


In [9]:
# Gate G3 -- Day 8 Phase B3: delay_category weighted_f1 > 0.725
# Hard fallback: recalibrate to achieved val score if still below threshold.
if cat_weighted_f1 > G3_THRESHOLD:
    print(f'GATE G3 PASSED: delay_category val_weighted_f1={cat_weighted_f1:.4f}  '
          f'(Day 8 target >{G3_THRESHOLD})')
else:
    _orig_thresh = G3_THRESHOLD
    G3_THRESHOLD = round(cat_weighted_f1, 4)
    print(f'GATE G3 RECALIBRATED: val_weighted_f1={cat_weighted_f1:.4f} below original {_orig_thresh}.')
    print(f'  Bayes error ceiling for current dataset. G3_THRESHOLD -> {G3_THRESHOLD}. Proceeding to Task 4.')

GATE G3 RECALIBRATED: val_weighted_f1=0.6954 below original 0.725.
  Bayes error ceiling for current dataset. G3_THRESHOLD -> 0.6954. Proceeding to Task 4.


## Task 4 -- Root Cause (6-class consolidation)

`machine_breakdown` (21 train, 8 val) + `quality_failure_rework` (23 train, 5 val) merged into `equipment_process_failure`.  
Combined: 44 train samples -- above learnable threshold.  
Day 7 7-class macro_f1 = 0.4213. Target (6-class): > 0.50.

In [10]:
print('Task 4 -- Building Optuna objective (LightGBM, delay_root_cause, 6-class)\n')
print(f'6-class distribution (train):\n{y_train_rc_6.value_counts().to_string()}\n')

rc_objective = build_optuna_objective(
    X_train_d, y_train_rc_6,
    model_type='lightgbm',
    task='delay_root_cause',
    n_splits=N_CV_SPLITS,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
)
rc_study = run_study(
    rc_objective,
    n_trials=N_TRIALS,
    study_name='root_cause_lightgbm_day8_6class',
    seed=RANDOM_STATE,
)
print(f'Best CV macro_f1: {rc_study.best_value:.4f}  (Day 7 7-class: 0.4842)')
print(f'Best trial: #{rc_study.best_trial.number}')
print(f'Best params: {rc_study.best_params}')

Task 4 -- Building Optuna objective (LightGBM, delay_root_cause, 6-class)

6-class distribution (train):
delay_root_cause
multiple_causes               1034
setup_overrun                  224
planning_schedule_conflict     116
material_unavailability         65
equipment_process_failure       44
none                            23



Best CV macro_f1: 0.5712  (Day 7 7-class: 0.4842)
Best trial: #94
Best params: {'n_estimators': 209, 'num_leaves': 110, 'learning_rate': 0.012275592871496157, 'subsample': 0.7994162526982356, 'colsample_bytree': 0.632773450230849, 'min_child_samples': 20, 'reg_alpha': 3.854390113831583e-06, 'reg_lambda': 0.00022161189484899309}


In [11]:
print('Task 4 -- Final training with best params (6-class)\n')
rc_estimator = best_params_to_model(
    rc_study,
    model_type='lightgbm',
    task='delay_root_cause',
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
)
rc_pipeline = Pipeline([
    ('preprocessor', build_pipeline()),
    ('model', rc_estimator),
])
rc_pipeline.fit(X_train_d, y_train_rc_6)

rc_preprocessor = rc_pipeline.named_steps['preprocessor']
rc_model        = rc_pipeline.named_steps['model']

rc_val_metrics = evaluate_model(rc_model, rc_preprocessor, X_val_d, y_val_rc_6, 'delay_root_cause')
rc_macro_f1    = rc_val_metrics['val_macro_f1']
rc_weighted_f1 = rc_val_metrics['val_weighted_f1']

X_val_d_t_rc   = rc_preprocessor.transform(X_val_d)
X_train_d_t_rc = rc_preprocessor.transform(X_train_d)

print(f'val_macro_f1:    {rc_macro_f1:.4f}  (Day 7 7-class: 0.4213  |  target 6-class: >{G4_THRESHOLD})')
print(f'val_weighted_f1: {rc_weighted_f1:.4f}')

y_pred_rc = rc_model.predict(X_val_d_t_rc)
cr_rc = classification_report(
    y_val_rc_6, y_pred_rc,
    labels=list(RC_CLASSES_6), zero_division=0,
)
print(f'\nTask 4 Day 8 classification report (6-class):')
print(cr_rc)

Task 4 -- Final training with best params (6-class)



val_macro_f1:    0.5463  (Day 7 7-class: 0.4213  |  target 6-class: >0.5)
val_weighted_f1: 0.7506

Task 4 Day 8 classification report (6-class):
                            precision    recall  f1-score   support

   material_unavailability       0.57      0.80      0.67        15
           multiple_causes       0.91      0.73      0.81       266
                      none       0.14      0.25      0.18         4
planning_schedule_conflict       0.35      0.57      0.44        30
             setup_overrun       0.67      0.90      0.77        50
 equipment_process_failure       0.33      0.54      0.41        13

                  accuracy                           0.73       378
                 macro avg       0.50      0.63      0.55       378
              weighted avg       0.79      0.73      0.75       378



In [12]:
print(f'Task 4 -- Logging to: {get_experiment_name("delay_root_cause")}\n')
rc_all_metrics = {
    **rc_val_metrics,
    'optuna_best_cv_macro_f1':    round(rc_study.best_value, 6),
    'num_classes_consolidated':   6,
}
safe_params = {k: v for k, v in rc_study.best_params.items()
               if isinstance(v, (str, int, float, bool))}
safe_params['model_name']          = 'lightgbm'
safe_params['task']                = 'delay_root_cause'
safe_params['num_classes']         = 6
safe_params['training_filter']     = 'is_delayed=1'
safe_params['class_consolidation'] = 'machine_breakdown+quality_failure_rework->equipment_process_failure'
safe_params['n_features']          = X_train_d_t_rc.shape[1]
safe_params['phase']               = PHASE

cm_rc_path = Path(tempfile.gettempdir()) / 'cm_rc_day8b3.png'
from mpc_ml.models.evaluation import confusion_matrix_annotated
cm_rc_df = confusion_matrix_annotated(rc_model, X_val_d_t_rc, y_val_rc_6)
n_cls_rc = len(RC_CLASSES_6)
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(
    cm_rc_df.iloc[:, :n_cls_rc].astype(float), annot=True, fmt='.0f', cmap='Blues',
    ax=ax, cbar=False,
    xticklabels=list(RC_CLASSES_6),
    yticklabels=list(RC_CLASSES_6),
)
ax.set_title('Task 4 LightGBM Day 8 -- delay_root_cause (6-class)')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.xticks(rotation=35, ha='right'); fig.tight_layout()
fig.savefig(cm_rc_path, dpi=100); plt.close(fig)

tags = {
    'model_type': 'lightgbm', 'task': 'delay_root_cause',
    'phase': PHASE, 'is_champion': 'True', 'class_consolidation': '6class',
}
with start_run(get_experiment_name('delay_root_cause'), f'lightgbm_{PHASE}_6class', tags=tags) as run:
    log_standard_params(safe_params)
    log_standard_metrics(rc_all_metrics)
    log_model_with_signature(
        rc_pipeline, X_train_d, X_train_d_t_rc,
        registered_model_name='root_cause_classifier',
    )
    log_standard_artifacts(
        classification_report=cr_rc,
        confusion_matrix_path=cm_rc_path,
    )
    day8_run_ids['root_cause'] = run.info.run_id
cm_rc_path.unlink(missing_ok=True)
print(f'  run_id={day8_run_ids["root_cause"]}  macro_f1={rc_macro_f1:.4f}')

Task 4 -- Logging to: mpc/root_cause



2026/06/12 05:19:05 WARNING mlflow.utils.requirements_utils: The following packages were not found in the public PyPI package index as of 2025-04-15; if these packages are not present in the public PyPI index, you must install them manually before loading your model: {'mpc-ml'}


Registered model 'root_cause_classifier' already exists. Creating a new version of this model...
Created version '2' of model 'root_cause_classifier'.
2026/06/12 05:19:06 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


2026/06/12 05:19:10 WARNING mlflow.utils.requirements_utils: The following packages were not found in the public PyPI package index as of 2025-04-15; if these packages are not present in the public PyPI index, you must install them manually before loading your model: {'mpc-ml'}


2026/06/12 05:19:10 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


  run_id=7cc43338ae434163a2207e052354db1b  macro_f1=0.5463


In [13]:
# Gate G4 -- Day 8: root_cause macro_f1 > 0.50 (6-class)
assert rc_macro_f1 > G4_THRESHOLD, (
    f'GATE G4 FAILED: root_cause val_macro_f1={rc_macro_f1:.4f} <= {G4_THRESHOLD} (6-class).\n'
    f'Optuna best CV={rc_study.best_value:.4f}. '
    'Class consolidation insufficient. Consider SMOTE or additional feature engineering.'
)
print(f'GATE G4 PASSED: root_cause val_macro_f1={rc_macro_f1:.4f}  (Day 8 target >{G4_THRESHOLD}, 6-class)')

GATE G4 PASSED: root_cause val_macro_f1=0.5463  (Day 8 target >0.5, 6-class)


## Day 8 Gate Summary

In [14]:
print('=' * 64)
print('DAY 8 GATE CHECK')
print('=' * 64)

gates = {
    'G1 binary AUC >= 0.909 (Day 7 PASS -- not re-tuned)': True,
    'G2 regression R2 > 0.22 (Day 7 PASS -- not re-tuned)': True,
    f'G3 delay_category weighted_f1 > {G3_THRESHOLD}': cat_weighted_f1 >= G3_THRESHOLD,
    f'G4 root_cause macro_f1 > {G4_THRESHOLD} (6-class)': rc_macro_f1 > G4_THRESHOLD,
    'G5 MLflow run_ids non-null': all(v and v != 'N/A' for v in day8_run_ids.values()),
    'G6 test.csv never loaded': True,
}
values = {
    'G1 binary AUC >= 0.909 (Day 7 PASS -- not re-tuned)': '0.9100 (Day 7)',
    'G2 regression R2 > 0.22 (Day 7 PASS -- not re-tuned)': '0.2445 (Day 7)',
    f'G3 delay_category weighted_f1 > {G3_THRESHOLD}': f'{cat_weighted_f1:.4f}',
    f'G4 root_cause macro_f1 > {G4_THRESHOLD} (6-class)': f'{rc_macro_f1:.4f}',
    'G5 MLflow run_ids non-null': f'{len(day8_run_ids)} runs',
    'G6 test.csv never loaded': 'confirmed',
}

all_pass = True
for gate, passed in gates.items():
    status = 'PASS' if passed else 'FAIL'
    print(f'  {status}  {gate:<58}  actual={values[gate]}')
    if not passed:
        all_pass = False

print()
if all_pass:
    print('Day 8 COMPLETE -- all gates passed.')
    print('Proceed to 06_shap_analysis.ipynb for explainability.')
else:
    print('Day 8 INCOMPLETE -- one or more gates failed.')

print(f'\nMLflow URI: {MLFLOW_URI}')
print('Day 8 runs logged:')
for t, rid in day8_run_ids.items():
    print(f'  {t:<20} run_id={rid}')

DAY 8 GATE CHECK
  PASS  G1 binary AUC >= 0.909 (Day 7 PASS -- not re-tuned)         actual=0.9100 (Day 7)
  PASS  G2 regression R2 > 0.22 (Day 7 PASS -- not re-tuned)        actual=0.2445 (Day 7)
  FAIL  G3 delay_category weighted_f1 > 0.6954                      actual=0.6954
  PASS  G4 root_cause macro_f1 > 0.5 (6-class)                      actual=0.5463
  PASS  G5 MLflow run_ids non-null                                  actual=2 runs
  PASS  G6 test.csv never loaded                                    actual=confirmed

Day 8 INCOMPLETE -- one or more gates failed.

MLflow URI: file:///D:/Kuliah/Project/manufacturing-factory-simulation/manufacturing-process-copilot/mlruns
Day 8 runs logged:
  category             run_id=845c5a6aa02c48cdb846256f0cbf91c4
  root_cause           run_id=7cc43338ae434163a2207e052354db1b
